<a href="https://colab.research.google.com/github/toche7/AI_ITM/blob/main/LabFinetuneSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Fine-tuning LLM with Unsloth using SQL-Create-Context Dataset

In [1]:
# Cell 1: ติดตั้ง Unsloth (ใช้เวลา ~3-5 นาที)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7xlmnos0/unsloth_8cba740de88441aeafd0bd33d0cc8f6b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7xlmnos0/unsloth_8cba740de88441aeafd0bd33d0cc8f6b
  Resolved https://github.com/unslothai/unsloth.git to commit 996b6a13d9176d173395dba382d904567027a57f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
  Running setup.py clean for xformers
Failed to build xformers
ERROR: ERROR: Failed to build installable w

In [1]:
# Cell 2: ตรวจสอบ GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# ควรเห็น: GPU: Tesla T4 | VRAM: 15.8 GB

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# Step 2
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/Meta-Llama-3.1-8B-Instruct",
    #max_seq_length = 2048,    # ความยาว context สูงสุด
    dtype         = None,     # auto-detect (bfloat16 บน T4)
    load_in_4bit  = True,     # QLoRA: quantize เป็น 4-bit NF4
)

# Inference test for the base model
print("\n--- Base Model Inference Test ---")
messages = [{
    "role": "user",
    "content": "\n\nHow many heads of the departments are older than 40 ?\n\nCREATE TABLE head "
}]

# Get the chat template for the tokenizer
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

model.generation_config.max_length = None


inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 512,
    temperature    = 0.7,      # ความ creative (0=deterministic, 1=สุ่ม)
    top_p          = 0.9,      # Nucleus sampling
    repetition_penalty = 1.1,  # ลดการพูดซ้ำ
)
print(tokenizer.decode(outputs[0][len(inputs[0]):]))
print("-----------------------------------")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Base Model Inference Test ---
**SQL Query to Count Heads of Departments Older Than 40**

```sql
SELECT COUNT(*) 
FROM head h
WHERE AGE(h.birth_date) > 40;
```

However, this query assumes that `birth_date` is a date field. If it's not, you may need to adjust the column name.

Assuming `head` table has columns like:

| Column Name | Data Type |
|-------------|-----------|
| id          | int       |
| department  | varchar   |
| birth_date  | date      |

Here is the detailed explanation:

1. We select all rows from the `head` table where:
2. The age of the person (calculated by subtracting their birth year from the current year) is greater than 40.

Note: This query uses SQL syntax and may vary depending on your database management system. For example, in MySQL, you would use `DATEDIFF(CURDATE(), birth_date)` instead of `AGE(birth_date)`.

Example Use Case:
Suppose we have the following data in our `head` table:

| id | department | birth_date |
|----|------------|------------|
| 

In [3]:
# Inference test for the base model
print("\n--- Base Model Inference Test ---")
messages = [{
    "role": "user",
    "content": "\n\nHow many heads of the departments are older than 50 ?\n\nCREATE TABLE head "
}]

# Get the chat template for the tokenizer
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
model.generation_config.max_length = None

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 512,
    temperature    = 0.7,      # ความ creative (0=deterministic, 1=สุ่ม)
    top_p          = 0.9,      # Nucleus sampling
    repetition_penalty = 1.1,  # ลดการพูดซ้ำ
)
print(tokenizer.decode(outputs[0][len(inputs[0]):]))
print("-----------------------------------")


--- Base Model Inference Test ---
To solve this, we'll need to assume a sample table `head` with some data. I will create such a table for demonstration purposes.

```sql
-- Create table and insert data
CREATE TABLE head (
    id INT,
    name VARCHAR(255),
    age INT,
    department_name VARCHAR(255)
);

INSERT INTO head (id, name, age, department_name) 
VALUES 
(1, 'John Smith', 60, 'Sales'),
(2, 'Jane Doe', 25, 'Marketing'),
(3, 'Bob Brown', 72, 'IT'),
(4, 'Alice Johnson', 40, 'HR');
```

Now, let's find out how many heads of departments are older than 50:

```sql
SELECT COUNT(*) 
FROM head 
WHERE age > 50;
```

This query counts the number of rows where the `age` is greater than 50. Since John Smith and Bob Brown are both over 50 years old, the result should be `2`. 

Please replace your own table structure and data into this SQL code snippet to get your desired answer. The table used here was created just for demonstration.<|eot_id|>
-----------------------------------


In [4]:
# Step 3
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 8,     # Reduced from 16 to save VRAM
    lora_alpha               = 16,    # Adjusted (2x r)
    lora_dropout             = 0.05,  # set zero some random input lora for protect overfting
    target_modules           = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention Layer
        "gate_proj", "up_proj", "down_proj"      # MLP Layer
    ],
    bias                     = "none", # Not train bias
    use_gradient_checkpointing = "unsloth", # Reduce vram
    random_state             = 3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.11 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.


In [5]:
# step 4
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# โหลด dataset ( format: question / context / answer)
dataset  = load_dataset("b-mc2/sql-create-context", split="train[:500]")
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

def format_alpaca(examples):
    """แปลง Alpaca format → chat messages → formatted text"""
    texts = []
    for inst, inp, out in zip(examples["question"],
                              examples["context"],
                              examples["answer"]):
        user_msg = inst if not inp else f"{inst}\n\n{inp}"
        convo = [
            {"role": "system",    "content": "You are a helpful assistant."},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": out},
        ]
        texts.append(tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

dataset    = dataset.map(format_alpaca, batched=True)
split      = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]   # 450 ตัวอย่าง
eval_data  = split["test"]    # 50 ตัวอย่าง
print(dataset[0]["text"][:300])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

How many heads of the departments are older than 56 ?

CREATE TABLE head (age INTEGER)<|eot_id|><|start_header_id|>assistant<|end_header_id|>

SELECT COUNT(*) 


In [6]:
print(dataset)

Dataset({
    features: ['answer', 'question', 'context', 'text'],
    num_rows: 500
})


In [7]:
import pandas as pd

df = dataset.to_pandas()

In [8]:
df.head(20)

,answer,question,context,text
0,SELECT COUNT(*) FROM head WHERE age > 56,How many heads of the departments are older th...,CREATE TABLE head (age INTEGER),<|begin_of_text|><|start_header_id|>system<|en...
1,"SELECT name, born_state, age FROM head ORDER B...","List the name, born state and age of the heads...","CREATE TABLE head (name VARCHAR, born_state VA...",<|begin_of_text|><|start_header_id|>system<|en...
2,"SELECT creation, name, budget_in_billions FROM...","List the creation year, name and budget of eac...","CREATE TABLE department (creation VARCHAR, nam...",<|begin_of_text|><|start_header_id|>system<|en...
3,"SELECT MAX(budget_in_billions), MIN(budget_in_...",What are the maximum and minimum budget of the...,CREATE TABLE department (budget_in_billions IN...,<|begin_of_text|><|start_header_id|>system<|en...
4,SELECT AVG(num_employees) FROM department WHER...,What is the average number of employees of the...,CREATE TABLE department (num_employees INTEGER...,<|begin_of_text|><|start_header_id|>system<|en...
5,SELECT name FROM head WHERE born_state <> 'Cal...,What are the names of the heads who are born o...,"CREATE TABLE head (name VARCHAR, born_state VA...",<|begin_of_text|><|start_header_id|>system<|en...
6,SELECT DISTINCT T1.creation FROM department AS...,What are the distinct creation years of the de...,"CREATE TABLE department (creation VARCHAR, dep...",<|begin_of_text|><|start_header_id|>system<|en...
7,SELECT born_state FROM head GROUP BY born_stat...,What are the names of the states where at leas...,CREATE TABLE head (born_state VARCHAR),<|begin_of_text|><|start_header_id|>system<|en...
8,SELECT creation FROM department GROUP BY creat...,In which year were most departments established?,CREATE TABLE department (creation VARCHAR),<|begin_of_text|><|start_header_id|>system<|en...
9,"SELECT T1.name, T1.num_employees FROM departme...",Show the name and number of employees for the ...,CREATE TABLE management (department_id VARCHAR...,<|begin_of_text|><|start_header_id|>system<|en...


In [9]:
import os
os.environ["WANDB_PROJECT"] = "my-finetuning"
# เพิ่ม report_to="wandb" ใน TrainingArguments

In [10]:
# Step 5
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_data,
    eval_dataset    = eval_data,
    dataset_text_field = "text",  # data column after formating
    max_seq_length  = 1024, # Reduced to 1024 to save VRAM on T4
    args = TrainingArguments(
        per_device_train_batch_size  = 1, # Reduced to 1 to avoid OOM
        gradient_accumulation_steps  = 8, # 1x8=8 effective batch
        num_train_epochs             = 3,
        learning_rate                = 2e-4,
        lr_scheduler_type            = "cosine", # learning rate adjusment pattern
        warmup_ratio                 = 0.05,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        eval_strategy                = "steps",
        eval_steps                   = 50,
        logging_steps                = 10,
        output_dir                   = "outputs",
        save_strategy                = "steps", # Match eval_strategy
        save_steps                   = 50,    # Save every 50 steps to pick best
        load_best_model_at_end       = True,
        metric_for_best_model        = "eval_loss",
    ),
)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-package

Step,Training Loss,Validation Loss
50,0.662368,0.592295
100,0.471478,0.493851
150,0.330109,0.489646
171,0.308826,0.488014


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-171/tokenizer_config.json.


TrainOutput(global_step=171, training_loss=0.5968232214102271, metrics={'train_runtime': 1100.5148, 'train_samples_per_second': 1.227, 'train_steps_per_second': 0.155, 'total_flos': 4583902956208128.0, 'train_loss': 0.5968232214102271})

In [11]:
# Step 6
FastLanguageModel.for_inference(model)  # เร็วขึ้น 2×
messages = [{
    "role": "user",
    "content": "\n\nHow many heads of the departments are older than 40 ?\n\nCREATE TABLE head "
}]

model.generation_config.max_length = None

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 512,
    temperature    = 0.7,      # ความ creative (0=deterministic, 1=สุ่ม)
    top_p          = 0.9,      # Nucleus sampling
    repetition_penalty = 1.1,  # ลดการพูดซ้ำ
)
print(tokenizer.decode(outputs[0][len(inputs[0]):]))

SELECT COUNT(*) FROM head WHERE age > 40<|eot_id|>


In [12]:
# Step 6
FastLanguageModel.for_inference(model)  # เร็วขึ้น 2×
messages = [{
    "role": "user",
    "content": "\n\nHow many heads of the departments are older than 50 ?\n\nCREATE TABLE head "
}]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

model.generation_config.max_length = None

outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 512,
    temperature    = 0.7,      # ความ creative (0=deterministic, 1=สุ่ม)
    top_p          = 0.9,      # Nucleus sampling
    repetition_penalty = 1.1,  # ลดการพูดซ้ำ
)
print(tokenizer.decode(outputs[0][len(inputs[0]):]))

SELECT COUNT(*) FROM head WHERE age > 50<|eot_id|>


In [7]:
# บันทึกเป็น GGUF (q4_k_m ≈ คุณภาพดี, ขนาดสมเหตุสมผล)
# เรากําหนดให้แปลงเป็น gguf ทันทีโดยระบุ quantization_method
model.save_pretrained_gguf(
    "my_model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

NameError: name 'model' is not defined

In [9]:
from llama_cpp import Llama

# โหลดโมเดล GGUF ที่แปลงเสร็จสมบูรณ์โดยตรง
model_path = "my_model_gguf_gguf/meta-llama-3.1-8b-instruct.Q4_K_M.gguf"
llm = Llama(model_path=model_path, n_ctx=512, n_gpu_layers=-1)

# ทดลองส่ง Prompt เพื่อสร้าง SQL
prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nHow many heads of the departments are older than 50 ?\n\nCREATE TABLE head <|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
response = llm(prompt, max_tokens=128, stop=["<|eot_id|>"])

print("\n--- Model Output ---")
print(response["choices"][0]["text"].strip())

llama_model_loader: loaded meta data with 34 key-value pairs and 292 tensors from my_model_gguf_gguf/meta-llama-3.1-8b-instruct.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.900000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   4:                               general.name str              = My_Model_Gguf
llama_model_loader: - kv   5:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   6:                         general.size_label str              = 8.0B
llama_model_loader: - kv   7:     


--- Model Output ---
SELECT COUNT(*) FROM head WHERE age > 50


In [11]:
prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nHow many heads of the departments are older than 40 ?\n\nCREATE TABLE head <|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
response = llm(prompt, max_tokens=128, stop=["<|eot_id|>"])

print("\n--- Model Output ---")
print(response["choices"][0]["text"].strip())

Llama.generate: 16 prefix-match hit, remaining 11 prompt tokens to eval
llama_perf_context_print:        load time =   11948.50 ms
llama_perf_context_print: prompt eval time =    3178.89 ms /    11 tokens (  288.99 ms per token,     3.46 tokens per second)
llama_perf_context_print:        eval time =    7280.66 ms /    10 runs   (  728.07 ms per token,     1.37 tokens per second)
llama_perf_context_print:       total time =   10467.88 ms /    21 tokens
llama_perf_context_print:    graphs reused =          9



--- Model Output ---
SELECT COUNT(*) FROM head WHERE Age > 40
